Analisis exploratio de los datos del dataset de entrenamiento
Para sabe rpor donde tirar, vamos a hacer 3 cosas:
- Comprobacion del balanceo de clases
- Visualizacion de los cuatro canales para varias muestras de cada clase (ej: 3)
- Limpieza de valores nulos ( NUNCA TIRANDO FILAS, nos cargamos la serie temproal entonces )


Cargamos el dataset como se dice en el readme.md del repo de la competicion (mirar el readme.md nustro)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from competition_api import load_dataset

In [ ]:
# 1. Cargar los datos usando la API oficial
print("Cargando datos...")
ds = load_dataset("data/train_100k.npz")
series = ds.series
labels = ds.labels

print(f"Dimensión de las series: {series.shape} -> (Muestras, Canales, Instantes de tiempo)")
print(f"Dimensión de las etiquetas: {labels.shape}\n")

In [ ]:
print(series[0, 0, :])  # Imprime la primera serie del primer canal

Ahora procedemos a hacer el balanceo de clases.

In [ ]:
print("--- BALANCE DE CLASES ---")
clases, conteos = np.unique(labels, return_counts=True)
for c, count in zip(clases, conteos):
    porcentaje = (count / len(labels)) * 100
    print(f"Clase {c}: {count} muestras ({porcentaje:.2f}%)")

Como podemos observar, ms o menos estan balanceadas, aunque de las 3 primeras hay un poco mas, y de la ultima la que menos, una diferencia del 6% ( no creo q suponga mucho la verdad, si estuvieran muy desbalancedas estariamos realmente jodidos )

Ahora vamos a por los valores nulos. Antes de ver que hacemos con ellos, hacemos un pequeño estudio de si se concentran en algun canal.

In [ ]:
nans_por_canal = np.isnan(series).sum(axis=(0, 2))
for i, nans in enumerate(nans_por_canal):
    print(f"Canal {i}: {nans} NaNs")

print("Sumna total de NaNs en todas las series:", np.isnan(series).sum())

Hay un total de 743076 valores nulos, y estan practicmanete divididos por igual en las 4 clases. Si bien parecen muchos, no confundirse pues es una matriz tridimensional. Tenemos 100000 muestras, pero en cada muestra, hay 256 lecturas de instantes de tiempo diferentes. Es decir, en total tenemos 25600000 de "datos". Por lo que no hay casi valores nulos. 

Para quitraos, como son -1%, basta con hacer una interpolacion ( recordemos, que como es una serie temporal el orden de las muestras tiene un significado, por lo que si nos falta la muestra 56, podemos hacer la media entre la 55 y la 57 para rellenar es ehueco de en medio)

Ahora vamos con la visualziacion

In [ ]:
# 4. Visualización de las Series Temporales
print("\nGenerando gráficas...")
fig, axes = plt.subplots(4, 2, figsize=(15, 12))
fig.suptitle('Ejemplo de Serie Temporal (4 Canales) para cada Clase', fontsize=16)

# Dibujar un ejemplo aleatorio para cada una de las 8 clases
for clase_id in range(8):
    # Coger todos los índices que pertenecen a esta clase
    indices_clase = np.where(labels == clase_id)[0]
    # Elegir uno al azar
    idx_aleatorio = np.random.choice(indices_clase)
    muestra = series[idx_aleatorio] # Forma (4, 256)
    
    fila = clase_id // 2
    col = clase_id % 2
    ax = axes[fila, col]
    
    # Dibujar los 4 canales de esa muestra
    for canal in range(4):
        ax.plot(muestra[canal, :], label=f'Canal {canal}')
        
    ax.set_title(f'Clase {clase_id} (Muestra {idx_aleatorio})')
    ax.set_xlabel('Tiempo (0-255)')
    ax.legend(loc='upper right', fontsize='small')

plt.tight_layout()
plt.show()

In [ ]:
tr = load_dataset("data/train_100k.npz")
te = load_dataset("data/test_demo.npz")

def gap_lengths(nan_mask, n=3000):
    """Longitud de los huecos de NaN consecutivos (canal 0 de n muestras)."""
    out = []
    for row in nan_mask[:n, 0]:
        d = np.diff(np.concatenate([[0], row.astype(int), [0]]))
        out += (np.where(d == -1)[0] - np.where(d == 1)[0]).tolist()
    return np.array(out)

def summary(ds, name):
    X = np.asarray(ds.series); nan = np.isnan(X)
    print(f"\n=== {name} {X.shape} ===")
    print("NaN %:", nan.mean() * 100)
    print("NaN por muestra (p50,p90,p99,max) %:", np.percentile(nan.mean((1, 2)) * 100, [50, 90, 99, 100]).round(2))
    print("muestras sin ningún NaN %:", (nan.sum((1, 2)) == 0).mean() * 100)
    print("canales enteros NaN:", nan.all(2).sum())
    g = gap_lengths(nan)
    if len(g): print("longitud de huecos (p50,p90,max):", np.percentile(g, [50, 90, 100]))
    print("inf:", np.isinf(X).sum(), " min/max:", np.nanmin(X), np.nanmax(X))
    print("media por canal:", np.nanmean(X, axis=(0, 2)).round(3))
    print("std por canal:  ", np.nanstd(X, axis=(0, 2)).round(3))
    print("mediana de std intra-muestra por canal:", np.median(np.nanstd(X, axis=2), axis=0).round(3))
    for k, v in ds.meta.items():
        u, c = np.unique(v, return_counts=True)
        print(f"meta[{k}] únicos={len(u)}:", dict(zip(u[:10].tolist(), (c / len(v))[:10].round(3))))

summary(tr, "train"); summary(te, "test_demo")

# ¿Hay fuga de información entre metadatos/NaN y etiqueta en train?
y = np.asarray(tr.labels); nanfrac = np.isnan(np.asarray(tr.series)).mean((1, 2))
print("\nNaN medio por clase:", [round(nanfrac[y == k].mean() * 100, 2) for k in range(8)])
for k, v in tr.meta.items():
    print(f"meta[{k}] por clase (media):", [round(float(np.mean(v[y == c])), 3) if np.issubdtype(v.dtype, np.number) else "-" for c in range(8)])

In [ ]:
tr = load_dataset("data/train_100k.npz")
te = load_dataset("data/test_demo.npz")
Xtr, Xte = np.asarray(tr.series), np.asarray(te.series)
ytr = np.asarray(tr.labels)

# 1) ¿Tiene etiquetas el demo?
print("labels demo:", np.unique(te.labels, return_counts=True))

# 2) Picos: cuantos y como son
for name, X in [("train", Xtr), ("demo", Xte)]:
    a = np.abs(X)
    mx = np.nanmax(a, axis=2)                       # (N,4) máximo por canal
    print(f"\n[{name}] |x| percentiles 50/99/99.9/99.99:", np.nanpercentile(a, [50, 99, 99.9, 99.99]).round(2))
    for t in (15, 30, 100):
        print(f"  muestras con algún |x|>{t}: {(mx.max(1) > t).mean()*100:.2f} %")
    # ¿cuántos puntos por muestra superan 15? (¿picos sueltos o ráfagas?)
    print("  puntos >15 por muestra afectada (p50,p90,max):",
          np.percentile((a > 15).sum((1, 2))[(a > 15).any((1, 2))], [50, 90, 100]) if (a > 15).any() else "ninguno")
    med = np.nanmedian(X, axis=2, keepdims=True)
    mad = np.nanmedian(np.abs(X - med), axis=2)
    print("  MAD mediano por canal:", np.median(mad, axis=0).round(3))

# 3) Número de NaN por muestra: histograma train vs demo
for name, X in [("train", Xtr), ("demo", Xte)]:
    n = np.isnan(X).sum((1, 2))
    print(f"[{name}] nº NaN por muestra percentiles 5/25/50/75/95:", np.percentile(n, [5, 25, 50, 75, 95]))

# 4) level: distribución y relación con la señal
lv_tr, lv_te = np.asarray(tr.meta["level"]), np.asarray(te.meta["level"])
print("\nlevel train percentiles:", np.percentile(lv_tr, [0, 10, 50, 90, 100]).round(2), " std", lv_tr.std().round(2))
print("level demo  percentiles:", np.percentile(lv_te, [0, 10, 50, 90, 100]).round(2), " std", lv_te.std().round(2))
print("level std por clase (train):", [round(float(lv_tr[ytr == k].std()), 2) for k in range(8)])
print("corr(level, media de la muestra) train:",
      np.corrcoef(lv_tr, np.nanmean(Xtr, (1, 2)))[0, 1].round(3), " demo:",
      np.corrcoef(lv_te, np.nanmean(Xte, (1, 2)))[0, 1].round(3))
if set(np.unique(te.labels)) - {-1}:
    yte = np.asarray(te.labels)
    print("level medio por clase (demo):", [round(float(lv_te[yte == k].mean()), 2) if (yte == k).any() else None for k in range(8)])

# 5) Espectro medio por sample_rate (¿cambia la escala temporal?) y train vs demo
def spec(X, mask):
    Z = np.nan_to_num(X[mask] - np.nanmean(X[mask], 2, keepdims=True))
    return np.log10((np.abs(np.fft.rfft(Z, axis=2)) ** 2).mean((0, 1)) + 1e-9)
sr_tr, sr_te = np.asarray(tr.meta["sample_rate"]), np.asarray(te.meta["sample_rate"])
bins = [1, 4, 8, 16, 32, 64, 128]
for name, X, sr in [("train", Xtr[:20000], sr_tr[:20000]), ("demo", Xte, sr_te)]:
    for s in (1.0, 2.0):
        sp = spec(X, sr == s)
        print(f"espectro log10 [{name}, sr={s}] bins {bins}:", sp[bins].round(2))

# 6) Dónde caen los NaN en el tiempo (inicio/fin) y por canal
for name, X in [("train", Xtr), ("demo", Xte)]:
    m = np.isnan(X).mean((0, 1))
    print(f"[{name}] NaN% en t=0..15, 120..135, 240..255:", (m[:16].mean()*100).round(2), (m[120:136].mean()*100).round(2), (m[240:].mean()*100).round(2))